# Prediccion del Mundial FIFA 2026

**Trabajo de Fin de Grado - Ingenieria del Software**

**Alumno: Raul Ramirez Adarve**

---

## Objetivo

Desarrollar un modelo de Machine Learning capaz de predecir los resultados del **Mundial 2026** (Estados Unidos, Mexico y Canada), simulando el torneo completo desde la fase de grupos hasta la final.

## Estructura

1. **Carga de datos** - Datasets de partidos internacionales desde 1872
2. **Analisis Exploratorio (EDA)** - Patrones y estadisticas clave
3. **Feature Engineering** - 40+ variables predictivas (ELO, forma, H2H, confederacion, rachas...)
4. **Modelado** - Comparacion de modelos, hyperparameter tuning, ensemble stacking
5. **Calibracion** - Calibracion de probabilidades para simulacion
6. **Simulacion del Mundial 2026** - Monte Carlo con 10,000 simulaciones
7. **Resultados** - Predicciones finales y visualizaciones

---
## 0. Configuracion e Imports

In [2]:
import sys
import os

# Añadir directorio raiz al path para importar modulos
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

from model.config import (
    RANDOM_SEED, FEATURE_COLS, TARGET, DATA_DIR,
    TRAIN_START_YEAR, WORLD_CUP_2026_GROUPS,
    MONTE_CARLO_SIMULATIONS,
)
from model.data.loader import load_results, load_goalscorers, load_shootouts
from model.features.engineering import build_features, prepare_splits, compute_sample_weights
from model.models.training import (
    compare_models, build_stacking_ensemble, calibrate_model,
    evaluate_on_test, save_model, TARGET_NAMES,
)
from model.models.tuning import tune_lightgbm, tune_xgboost
from model.simulation.simulator import WorldCupSimulator
from model.visualization.plots import (
    setup_style, plot_result_distribution, plot_temporal_evolution,
    plot_elo_evolution, plot_feature_correlations, plot_model_comparison,
    plot_confusion_matrix, plot_feature_importance, plot_calibration,
    plot_monte_carlo_results, plot_tournament_progression,
    plot_group_predictions,
)

np.random.seed(RANDOM_SEED)
setup_style()

print(f'Features configuradas: {len(FEATURE_COLS)}')
print(f'Simulaciones Monte Carlo: {MONTE_CARLO_SIMULATIONS:,}')

Features configuradas: 47
Simulaciones Monte Carlo: 10,000


---
## 1. Carga de Datos

In [3]:
# Cargar datasets
df = load_results()
goalscorers = load_goalscorers()
shootouts = load_shootouts()

print(f'Partidos totales: {len(df):,}')
print(f'Periodo: {df["date"].min().date()} a {df["date"].max().date()}')
print(f'Equipos unicos: {df["home_team"].nunique()}')
print(f'Torneos unicos: {df["tournament"].nunique()}')
print(f'\nGoleadores: {len(goalscorers):,} registros')
print(f'Tandas de penaltis: {len(shootouts):,} registros')
print(f'\nPrimeros 5 partidos:')
df.tail(5)[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']]

Partidos totales: 49,071
Periodo: 1872-11-30 a 2026-01-26
Equipos unicos: 325
Torneos unicos: 191

Goleadores: 47,555 registros
Tandas de penaltis: 665 registros

Primeros 5 partidos:


,date,home_team,away_team,home_score,away_score,tournament
49066,2026-01-18,Grenada,Jamaica,0,1,Friendly
49067,2026-01-18,Morocco,Senegal,0,1,African Cup of Nations
49068,2026-01-22,Panama,Mexico,0,1,Friendly
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly
49070,2026-01-26,Uzbekistan,China PR,2,2,Friendly


---
## 2. Analisis Exploratorio de Datos (EDA)

In [4]:
# Distribucion de resultados
print('Distribucion de resultados:')
print(f'  Victoria Local:    {(df["result"]==2).sum():>6,} ({(df["result"]==2).mean()*100:.1f}%)')
print(f'  Empate:            {(df["result"]==1).sum():>6,} ({(df["result"]==1).mean()*100:.1f}%)')
print(f'  Victoria Visitante:{(df["result"]==0).sum():>6,} ({(df["result"]==0).mean()*100:.1f}%)')

plot_result_distribution(df)

Distribucion de resultados:
  Victoria Local:    24,043 (49.0%)
  Empate:            11,156 (22.7%)
  Victoria Visitante:13,872 (28.3%)


In [5]:
# Evolucion temporal
plot_temporal_evolution(df)

# Top torneos
print('\nTop 10 torneos por numero de partidos:')
top_tournaments = df['tournament'].value_counts().head(10)
for t, n in top_tournaments.items():
    print(f'  {t:45s} {n:>5,} ({n/len(df)*100:.1f}%)')


Top 10 torneos por numero de partidos:
  Friendly                                      18,181 (37.1%)
  FIFA World Cup qualification                  8,755 (17.8%)
  UEFA Euro qualification                       2,824 (5.8%)
  African Cup of Nations qualification          2,327 (4.7%)
  FIFA World Cup                                  964 (2.0%)
  Copa América                                    869 (1.8%)
  African Cup of Nations                          845 (1.7%)
  AFC Asian Cup qualification                     829 (1.7%)
  UEFA Nations League                             658 (1.3%)
  CECAFA Cup                                      620 (1.3%)


In [6]:
# Ventaja de jugar en casa vs campo neutral
home_wins_home = (df[~df['neutral']]['result'] == 2).mean() * 100
home_wins_neutral = (df[df['neutral']]['result'] == 2).mean() * 100

print(f'Porcentaje de victoria local:')
print(f'  En campo propio:  {home_wins_home:.1f}%')
print(f'  En campo neutral: {home_wins_neutral:.1f}%')
print(f'  Diferencia:       {home_wins_home - home_wins_neutral:.1f} puntos porcentuales')
print(f'\n  -> En el Mundial 2026 todos los partidos seran en campo neutral')

Porcentaje de victoria local:
  En campo propio:  50.7%
  En campo neutral: 44.2%
  Diferencia:       6.6 puntos porcentuales

  -> En el Mundial 2026 todos los partidos seran en campo neutral


In [7]:
# Estadisticas de goles
print('Estadisticas de goles por partido:')
total_goals = df['home_score'] + df['away_score']
print(f'  Media de goles por partido: {total_goals.mean():.2f}')
print(f'  Mediana: {total_goals.median():.0f}')
print(f'  Partidos 0-0: {(total_goals == 0).sum():,} ({(total_goals == 0).mean()*100:.1f}%)')
print(f'  Partidos con 5+ goles: {(total_goals >= 5).sum():,} ({(total_goals >= 5).mean()*100:.1f}%)')

Estadisticas de goles por partido:
  Media de goles por partido: 2.94
  Mediana: 3
  Partidos 0-0: 3,943 (8.0%)
  Partidos con 5+ goles: 9,241 (18.8%)


---
## 3. Feature Engineering

Generamos 40+ features agrupadas en:
- **ELO** (4): Sistema de rating propio con pesos por torneo
- **Forma reciente** (14): Ultimos 10 partidos de cada equipo
- **Head-to-Head** (6): Ultimos 5 enfrentamientos directos
- **Diferencias** (3): Diferencia de forma entre equipos
- **Contextuales** (3): Campo neutral, peso torneo, competitividad
- **Confederacion** (3): Fortaleza de confederacion, inter-confederacion
- **Rachas** (4): Racha de victorias/invictos
- **Descanso** (3): Dias desde ultimo partido
- **Clean sheets** (2): Porcentaje porteria a 0
- **Fase torneo** (1): Eliminatoria vs grupos
- **Experiencia** (2): Partidos en grandes torneos
- **Local/Visitante** (2): Rendimiento especifico como local/visitante

In [8]:
%%time
# Generar todas las features (procesamiento secuencial cronologico)
print('Generando features...')
df = build_features(df)
print(f'\nFeatures generadas: {len(FEATURE_COLS)}')
print(f'Dataset completo: {len(df):,} partidos')

# Mostrar features
for i, col in enumerate(FEATURE_COLS, 1):
    print(f'  {i:2d}. {col}')

Generando features...

Features generadas: 47
Dataset completo: 49,071 partidos
   1. home_elo
   2. away_elo
   3. elo_diff
   4. elo_expected_home
   5. home_form_wins
   6. home_form_draws
   7. home_form_losses
   8. home_form_gf
   9. home_form_gc
  10. home_form_points
  11. home_form_gd
  12. away_form_wins
  13. away_form_draws
  14. away_form_losses
  15. away_form_gf
  16. away_form_gc
  17. away_form_points
  18. away_form_gd
  19. h2h_home_wins
  20. h2h_away_wins
  21. h2h_draws
  22. h2h_home_goals
  23. h2h_away_goals
  24. h2h_dominance
  25. form_points_diff
  26. form_gd_diff
  27. form_gf_diff
  28. neutral_int
  29. tournament_weight
  30. is_competitive
  31. home_conf_strength
  32. away_conf_strength
  33. is_inter_confederation
  34. home_streak
  35. away_streak
  36. home_unbeaten_streak
  37. away_unbeaten_streak
  38. home_days_rest
  39. away_days_rest
  40. rest_advantage
  41. home_clean_sheet_pct
  42. away_clean_sheet_pct
  43. is_knockout
  44. home_ma

In [9]:
# Guardar trackers para simulacion
elo_system = df.attrs.get('elo_system')
form_tracker = df.attrs.get('form_tracker')
h2h_tracker = df.attrs.get('h2h_tracker')
major_exp = df.attrs.get('major_exp', {})
conf_wins = df.attrs.get('conf_wins', {})

# Mostrar top ELOs actuales
if elo_system:
    teams_elo = [(t, elo_system.get(t)) for t in set(df['home_team'].unique())]
    teams_elo.sort(key=lambda x: x[1], reverse=True)
    print('Top 20 selecciones por ELO actual:')
    print(f'{"Pos":>4s}  {"Equipo":25s} {"ELO":>6s}')
    print('-' * 40)
    for i, (team, elo) in enumerate(teams_elo[:20], 1):
        print(f'{i:4d}  {team:25s} {elo:6.0f}')

Top 20 selecciones por ELO actual:
 Pos  Equipo                       ELO
----------------------------------------
   1  Spain                       2262
   2  Argentina                   2189
   3  France                      2135
   4  England                     2129
   5  Colombia                    2093
   6  Brazil                      2069
   7  Ecuador                     2039
   8  Netherlands                 2037
   9  Portugal                    2030
  10  Germany                     2003
  11  Croatia                     2001
  12  Senegal                     1998
  13  Norway                      1997
  14  Switzerland                 1994
  15  Japan                       1985
  16  Uruguay                     1980
  17  Mexico                      1966
  18  Turkey                      1958
  19  Paraguay                    1940
  20  Belgium                     1934


In [10]:
# Evolucion ELO de selecciones clave
top_teams = ['Brazil', 'Argentina', 'France', 'Germany', 'Spain',
             'England', 'Netherlands', 'Portugal', 'Italy', 'Belgium']
plot_elo_evolution(df, top_teams)

---
## 4. Preparacion de Datos y Correlaciones

In [11]:
# Filtrar desde 2000 y crear splits
splits = prepare_splits(df)

print('Division temporal de datos:')
print(f'{"Set":15s} {"Partidos":>10s} {"Local%":>8s} {"Emp%":>7s} {"Vis%":>7s}')
print('-' * 55)
for name in ['train', 'val', 'test', 'wc2022']:
    X = splits[f'X_{name}']
    y = splits[f'y_{name}']
    label = 'Mundial 2022' if name == 'wc2022' else name.capitalize()
    print(f'{label:15s} {len(X):10,} {(y==2).mean()*100:7.1f}% '
          f'{(y==1).mean()*100:6.1f}% {(y==0).mean()*100:6.1f}%')

Division temporal de datos:
Set               Partidos   Local%    Emp%    Vis%
-------------------------------------------------------
Train               20,743    48.2%   23.4%   28.4%
Val                    905    50.2%   22.7%   27.2%
Test                 3,301    47.0%   22.8%   30.2%
Mundial 2022            64    43.8%   23.4%   32.8%


In [12]:
# Correlaciones con target
df_model = splits['df_model']
plot_feature_correlations(df_model, FEATURE_COLS, TARGET)

In [13]:
# Calcular sample weights
sample_weight = compute_sample_weights(splits['df_train'], splits['y_train'])
print(f'Sample weights calculados: min={sample_weight.min():.3f}, '
      f'max={sample_weight.max():.3f}, mean={sample_weight.mean():.3f}')

Sample weights calculados: min=0.197, max=3.753, mean=1.000


---
## 5. Modelado

### 5.1 Comparacion de modelos base

In [14]:
%%time
print('=== COMPARACION DE MODELOS BASE ===\n')
results = compare_models(
    splits['X_train'], splits['y_train'],
    splits['X_val'], splits['y_val'],
    splits['X_train_sc'], splits['X_val_sc'],
    sample_weight=sample_weight,
)

plot_model_comparison(results)

=== COMPARACION DE MODELOS BASE ===

  Baseline (siempre Local)            | Train: 0.4822 | Val: 0.5017 | LogLoss: 17.9621
  Regresion Logistica                 | Train: 0.5490 | Val: 0.5050 | LogLoss: 0.9709
  Random Forest                       | Train: 0.6310 | Val: 0.5271 | LogLoss: 0.9483
  XGBoost                             | Train: 0.6138 | Val: 0.5724 | LogLoss: 0.9143
  LightGBM                            | Train: 0.6426 | Val: 0.5646 | LogLoss: 0.9218
CPU times: total: 54.5 s
Wall time: 20.5 s


### 5.2 Hyperparameter Tuning con Optuna

In [15]:
%%time
# Tuning de LightGBM (reducir n_trials para ejecucion rapida, aumentar para mejor resultado)
lgbm_params, lgbm_tuned = tune_lightgbm(
    splits['X_train'], splits['y_train'],
    sample_weight=sample_weight,
    n_trials=50,  # Aumentar a 100+ para mejor resultado
)

Optimizando LightGBM con 50 trials...


Best trial: 22. Best value: 0.919115: 100%|██████████| 50/50 [11:06<00:00, 13.33s/it]


  Mejor log_loss CV: 0.9191
  Mejores parametros: {'n_estimators': 155, 'max_depth': 7, 'learning_rate': 0.023817338843310245, 'subsample': 0.6765295679257393, 'colsample_bytree': 0.5769615508241345, 'min_child_samples': 17, 'reg_alpha': 0.0034720473886603316, 'reg_lambda': 0.031939969404994134, 'num_leaves': 21, 'class_weight': 'balanced', 'random_state': 42, 'verbosity': -1}
CPU times: total: 50min 26s
Wall time: 11min 8s


In [16]:
%%time
# Tuning de XGBoost
xgb_params, xgb_tuned = tune_xgboost(
    splits['X_train'], splits['y_train'],
    sample_weight=sample_weight,
    n_trials=50,
)

Optimizando XGBoost con 50 trials...


  0%|          | 0/50 [00:11<?, ?it/s]


[W 2026-02-26 13:23:59,196] Trial 0 failed with parameters: {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.01735661101737246, 'subsample': 0.8387375029672205, 'colsample_bytree': 0.9278514030316429, 'min_child_weight': 7, 'reg_alpha': 1.0435513534254548, 'reg_lambda': 0.005983554600521416, 'gamma': 0.08394650856921727} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\adarv\Escritorio\INSO\INSO4\TFG\world-cup-predictor\venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "c:\Users\adarv\Escritorio\INSO\INSO4\TFG\world-cup-predictor\src\models\tuning.py", line 130, in <lambda>
    lambda trial: _xgb_objective(trial, X_train, y_train, sample_weight),
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\adarv\Escritorio\INSO\INSO4\TFG\world-cup-predictor\src\models\tuning.py", line 80, in _xgb

KeyboardInterrupt: 

In [ ]:
# Evaluar modelos tuneados en validation
from model.models.training import train_and_evaluate

print('\n=== MODELOS TUNEADOS ===\n')

lgbm_result = train_and_evaluate(
    'LightGBM Tuned', lgbm_tuned,
    splits['X_train'], splits['y_train'],
    splits['X_val'], splits['y_val'],
    sample_weight=sample_weight,
)
print(f"  LightGBM Tuned: Val Acc={lgbm_result['val_acc']:.4f}, LogLoss={lgbm_result['val_logloss']:.4f}")

xgb_result = train_and_evaluate(
    'XGBoost Tuned', xgb_tuned,
    splits['X_train'], splits['y_train'],
    splits['X_val'], splits['y_val'],
    sample_weight=sample_weight,
)
print(f"  XGBoost Tuned:  Val Acc={xgb_result['val_acc']:.4f}, LogLoss={xgb_result['val_logloss']:.4f}")

### 5.3 Ensemble Stacking

In [ ]:
%%time
print('=== STACKING ENSEMBLE ===\n')
stacking_result = build_stacking_ensemble(
    splits['X_train'], splits['y_train'],
    splits['X_val'], splits['y_val'],
    sample_weight=sample_weight,
)

In [ ]:
# Seleccionar mejor modelo
all_results = results + [lgbm_result, xgb_result, stacking_result]
all_results.sort(key=lambda x: x['val_logloss'])  # Ordenar por log_loss (menor = mejor)

print('\n=== RANKING FINAL (por Log Loss) ===\n')
print(f'{"Pos":>4s}  {"Modelo":35s} {"Val Acc":>9s} {"Val LogLoss":>12s}')
print('-' * 65)
for i, r in enumerate(all_results, 1):
    marker = ' <-- MEJOR' if i == 1 else ''
    print(f'{i:4d}  {r["name"]:35s} {r["val_acc"]:8.4f} {r["val_logloss"]:11.4f}{marker}')

best = all_results[0]
best_model = best['model']
print(f'\nMejor modelo seleccionado: {best["name"]}')

### 5.4 Calibracion de Probabilidades

In [ ]:
# Calibrar el mejor modelo
print('Calibrando probabilidades del mejor modelo...')
calibrated_model = calibrate_model(
    best_model, splits['X_train'], splits['y_train'],
    method='isotonic', cv=5,
)

# Evaluar calibracion en validation
y_val_proba_calib = calibrated_model.predict_proba(splits['X_val'])
from sklearn.metrics import log_loss, accuracy_score
y_val_pred_calib = calibrated_model.predict(splits['X_val'])

print(f'\nAntes de calibrar:  Acc={best["val_acc"]:.4f}, LogLoss={best["val_logloss"]:.4f}')
print(f'Despues de calibrar: Acc={accuracy_score(splits["y_val"], y_val_pred_calib):.4f}, '
      f'LogLoss={log_loss(splits["y_val"], y_val_proba_calib):.4f}')

# Diagrama de calibracion
plot_calibration(splits['y_val'], y_val_proba_calib)

---
## 6. Evaluacion en Test

### 6.1 Test general (2023+)

In [ ]:
# Evaluar en test set (2023+)
test_eval = evaluate_on_test(
    calibrated_model, splits['X_test'], splits['y_test'],
    dataset_name='Test (2023+)',
)

plot_confusion_matrix(
    splits['y_test'], test_eval['y_pred'],
    title='- Test (2023+)',
    filename='10_matriz_confusion_test.png',
)

### 6.2 Evaluacion especifica: Mundial 2022

In [ ]:
# Evaluar en Mundial 2022
if len(splits['X_wc2022']) > 0:
    wc2022_eval = evaluate_on_test(
        calibrated_model, splits['X_wc2022'], splits['y_wc2022'],
        dataset_name='Mundial 2022',
    )
    
    plot_confusion_matrix(
        splits['y_wc2022'], wc2022_eval['y_pred'],
        title='- Mundial 2022',
        filename='10b_mundial_2022_confusion.png',
    )
    
    # Detalle partido a partido
    df_wc = splits['df_wc2022'].copy()
    df_wc['pred'] = wc2022_eval['y_pred']
    df_wc['correct'] = df_wc['result'] == df_wc['pred']
    
    proba = wc2022_eval['y_proba']
    print(f'\nDetalle de predicciones del Mundial 2022:')
    print(f'{"":>3s} {"Partido":43s} {"Real":18s} {"Predicho":18s} {"P(Local)":>9s} {"P(Emp)":>8s} {"P(Vis)":>9s}')
    print('-' * 110)
    for idx, (_, row) in enumerate(df_wc.iterrows()):
        real_map = {0: 'Victoria Visitante', 1: 'Empate', 2: 'Victoria Local'}
        partido = f"{row['home_team']} vs {row['away_team']}"
        mark = 'V' if row['correct'] else 'X'
        print(f'{mark:>3s} {partido:43s} {real_map[row["result"]]:18s} '
              f'{real_map[row["pred"]]:18s} {proba[idx,2]:8.1%} {proba[idx,1]:7.1%} {proba[idx,0]:8.1%}')
    
    print(f'\nAciertos: {df_wc["correct"].sum()}/{len(df_wc)} ({df_wc["correct"].mean()*100:.1f}%)')
else:
    print('No hay datos del Mundial 2022 disponibles')

In [ ]:
# Feature importance
# Usar el modelo base (no calibrado) para feature importance
plot_feature_importance(best_model, FEATURE_COLS, top_n=25)

In [ ]:
# Guardar modelo
save_model(calibrated_model, 'best_calibrated_model.joblib')
save_model(best_model, 'best_model.joblib')

---
## 7. Simulacion del Mundial 2026

### 7.1 Grupos del Mundial 2026

In [ ]:
# Mostrar grupos
print('=== GRUPOS DEL MUNDIAL 2026 ===\n')
for group_name, teams in sorted(WORLD_CUP_2026_GROUPS.items()):
    elos = [(t, elo_system.get(t)) for t in teams]
    elos.sort(key=lambda x: x[1], reverse=True)
    print(f'Grupo {group_name}:')
    for t, e in elos:
        print(f'  {t:25s} (ELO: {e:.0f})')
    print()

### 7.2 Simulacion Monte Carlo

In [ ]:
%%time
# Crear simulador
simulator = WorldCupSimulator(
    model=calibrated_model,
    elo_system=elo_system,
    form_tracker=form_tracker,
    h2h_tracker=h2h_tracker,
    major_exp=major_exp,
    conf_wins=conf_wins,
)

# Ejecutar Monte Carlo
mc_results = simulator.monte_carlo(n_simulations=MONTE_CARLO_SIMULATIONS)

In [ ]:
# Visualizaciones de resultados
plot_monte_carlo_results(mc_results, top_n=20)
plot_tournament_progression(mc_results, top_n=20)

### 7.3 Prediccion determinista (partido a partido)

In [ ]:
# Simulacion determinista (1 corrida) para ver el bracket
det_result = simulator.simulate_tournament(rng=np.random.default_rng(RANDOM_SEED))

# Mostrar resultados de fase de grupos
print('=== FASE DE GRUPOS (prediccion) ===\n')
for group_name in sorted(det_result['group_standings'].keys()):
    standings = det_result['group_standings'][group_name]
    print(f'Grupo {group_name}:')
    print(f'  {"Equipo":25s} {"PJ":>3s} {"PG":>3s} {"PE":>3s} {"PP":>3s} '
          f'{"GF":>3s} {"GC":>3s} {"DG":>4s} {"Pts":>4s}')
    for _, row in standings.iterrows():
        marker = '*' if row['position'] <= 2 else (' ' if row['position'] == 3 else ' ')
        print(f'{marker} {row["team"]:25s} {row["played"]:3.0f} {row["wins"]:3.0f} '
              f'{row["draws"]:3.0f} {row["losses"]:3.0f} {row["gf"]:3.0f} {row["gc"]:3.0f} '
              f'{row["gd"]:+4.0f} {row["points"]:4.0f}')
    print()

plot_group_predictions(det_result['group_standings'])

In [ ]:
# Mostrar bracket de eliminatorias
print('=== ELIMINATORIAS ===\n')

for round_name, display_name in [('r32', 'Ronda de 32'), ('r16', 'Octavos de Final'),
                                   ('qf', 'Cuartos de Final'), ('sf', 'Semifinales'),
                                   ('final', 'FINAL')]:
    matches = det_result['results'].get(round_name, [])
    if matches:
        print(f'--- {display_name} ---')
        for m in matches:
            winner_marker = lambda t: ' (*)' if t == m['winner'] else ''
            print(f"  {m['home_team']}{winner_marker(m['home_team'])} {m['home_score']}-{m['away_score']} "
                  f"{m['away_team']}{winner_marker(m['away_team'])}")
        print()

print(f'\n{"="*50}')
print(f'CAMPEON DEL MUNDO 2026: {det_result["champion"]}')
print(f'Subcampeon: {det_result["runner_up"]}')
print(f'{"="*50}')

### 7.4 Resumen completo de probabilidades

In [ ]:
# Tabla completa de probabilidades
print('=== PROBABILIDADES COMPLETAS - MUNDIAL 2026 ===\n')
print(f'{"Pos":>4s}  {"Equipo":25s} {"ELO":>6s} {"Campeon":>8s} {"Final":>7s} '
      f'{"Semis":>7s} {"Cuartos":>8s} {"R16":>6s}')
print('-' * 80)
for i, row in mc_results.iterrows():
    if row['champion_pct'] > 0 or i < 48:
        print(f'{i+1:4d}  {row["team"]:25s} {row["elo"]:6.0f} {row["champion_pct"]:7.1f}% '
              f'{row["final_pct"]:6.1f}% {row["sf_pct"]:6.1f}% '
              f'{row["qf_pct"]:7.1f}% {row["r16_pct"]:5.1f}%')

# Guardar resultados
mc_results.to_csv(str(DATA_DIR.parent / 'outputs' / 'world_cup_2026_predictions.csv'), index=False)
print(f'\nResultados guardados en outputs/world_cup_2026_predictions.csv')

---
## 8. Conclusiones

### Resumen del modelo
- **Features**: 40+ variables predictivas (ELO, forma, H2H, confederacion, rachas, etc.)
- **Modelo**: Ensemble calibrado de modelos gradient boosting
- **Simulacion**: 10,000 iteraciones Monte Carlo del torneo completo

### Limitaciones
- No incluye datos de jugadores individuales (lesiones, estado de forma)
- No considera factores tacticos ni cambios de entrenador recientes
- Los equipos de repesca usan el favorito como placeholder
- El formato 2026 (48 equipos) es nuevo, sin precedente historico directo